### 전체 코드 설명 개요

이 노트북은 Google Colab 환경에서 동영상 파일에 '야간 효과(Night Effect)'를 일괄적으로 적용하기 위한 파이썬 스크립트입니다. '야간 효과'는 감마 보정(어둡게 하기)과 노이즈 추가를 통해 구현됩니다. 이 과정은 `apply_night_effect` 함수에 의해 단일 동영상에 적용되며, `batch_convert_videos` 함수를 사용하여 여러 동영상을 병렬로 처리합니다.

### 1단계: 환경 설정 및 라이브러리 임포트
이 섹션에서는 영상 처리에 필요한 라이브러리를 불러오고, 구글 드라이브를 연결합니다.

In [ ]:
import cv2
import numpy as np
import os
import random
import subprocess
from google.colab import drive

# Google Drive 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("Google Drive가 성공적으로 마운트되었습니다.")
else:
    print("Google Drive가 이미 마운트되어 있습니다.")

print("필요한 라이브러리 임포트 및 환경 설정이 완료되었습니다.")

Mounted at /content/drive
Google Drive가 성공적으로 마운트되었습니다.
필요한 라이브러리 임포트 및 환경 설정이 완료되었습니다.


#### 2단계: 야간 효과 적용 함수 정의 (`apply_night_effect`) (셀 2e1ed3ad)

이 함수는 개별 동영상 파일에 야간 효과를 적용하는 핵심 로직을 포함하고 있습니다.

*   **주요 기능:**
    *   `cv2`, `numpy`, `os`, `random`, `subprocess` 라이브러리를 임포트합니다.
    *   `gamma`와 `noise_sigma` 값을 무작위로 생성하여 각 동영상마다 약간씩 다른 야간 효과를 적용합니다.
    *   **동영상 로드:** `cv2.VideoCapture`를 사용하여 `input_path`에서 동영상을 로드하고, 프레임 너비, 높이, FPS를 가져옵니다.
    *   **비트레이트 계산:** `ffprobe`를 사용하여 원본 동영상의 비트레이트를 추출합니다. `bitrate_factor`를 곱하여 출력 동영상의 비트레이트를 조절합니다. `ffprobe`가 실패할 경우, 파일 크기 기반으로 비트레이트를 추정하거나 기본값을 사용합니다.
    *   **FFmpeg 파이프 설정:** `subprocess.Popen`을 사용하여 FFmpeg 프로세스를 실행하고, 처리된 프레임을 FFmpeg의 표준 입력(`stdin`)으로 보내 동영상을 인코딩합니다. `libx264` 코덱과 계산된 비트레이트를 사용하여 용량과 품질의 균형을 맞춥니다.
    *   **감마 보정 (Gamma Correction):** `lut` (룩업 테이블)을 생성하여 각 픽셀 값에 감마 보정을 적용하여 이미지를 어둡게 만듭니다.
    *   **노이즈 추가 (Noise Addition):** 무작위로 생성된 `noise_pool`을 사용하여 각 프레임에 노이즈를 추가하여 야간 시야와 유사한 효과를 냅니다.
    *   **프레임 처리:** 동영상의 각 프레임을 읽어와 감마 보정 및 노이즈를 적용한 후, FFmpeg 파이프로 전송합니다.
    *   **오류 처리:** 동영상 파일을 열 수 없거나 FFmpeg 인코딩 중 오류가 발생할 경우, 관련 오류 메시지를 반환합니다.

In [ ]:
import cv2
import numpy as np
import os
import random
import subprocess

def apply_night_effect(input_path, output_path, bitrate_factor=1.0):
    gamma = random.uniform(3.5, 5.0)
    noise_sigma = random.uniform(12, 25)

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        return False, f"영상 파일을 열 수 없습니다: {input_path}"

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    if width <= 0 or height <= 0 or fps <= 0 or total_frames <= 0:
        cap.release()
        return False, f"유효하지 않은 영상 메타데이터 (W:{width}, H:{height}, FPS:{fps})"

    ffmpeg_input_fps = str(int(fps)) if fps % 1 == 0 else str(fps)

    initial_bitrate_k = 2000
    calculated_bitrate_bps = None
    try:
        cmd_bitrate = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', input_path]
        bitrate_output = subprocess.check_output(cmd_bitrate, stderr=subprocess.DEVNULL).decode().strip()
        if bitrate_output and bitrate_output != 'N/A':
            calculated_bitrate_bps = int(bitrate_output)
    except:
        pass

    if calculated_bitrate_bps is None:
        try:
            duration_seconds = total_frames / fps
            calculated_bitrate_bps = (os.path.getsize(input_path) * 8) / duration_seconds
        except:
            calculated_bitrate_bps = initial_bitrate_k * 1000

    final_bitrate_k = max(int(calculated_bitrate_bps * bitrate_factor / 1000), 500)
    bitrate_str = f'{final_bitrate_k}k'

    # GPU 오류 해결을 위해 CPU 인코더(libx264)로 변경
    ffmpeg_cmd = [
        'ffmpeg', '-y', '-f', 'rawvideo', '-vcodec', 'rawvideo',
        '-s', f'{width}x{height}', '-pix_fmt', 'bgr24', '-r', ffmpeg_input_fps,
        '-i', '-',
        '-c:v', 'libx264', # 하드웨어 제한이 없는 CPU 인코더 사용
        '-b:v', bitrate_str,
        '-maxrate', bitrate_str,
        '-bufsize', f'{final_bitrate_k * 2}k',
        '-pix_fmt', 'yuv420p',
        '-preset', 'ultrafast', # 처리 속도 확보
        '-movflags', 'faststart',
        '-threads', '1', # 프로세스당 1개 스레드 할당 (병렬 처리 최적화)
        output_path
    ]

    proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE)

    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype("uint8")
    noise_pool = []
    for _ in range(30):
        temp_noise = np.zeros((height, width, 3), dtype=np.float32)
        cv2.randn(temp_noise, np.array([0, 0, 0]), np.array([noise_sigma, noise_sigma, noise_sigma]))
        noise_pool.append((np.clip(temp_noise, 0, 255).astype(np.uint8), np.clip(-temp_noise, 0, 255).astype(np.uint8)))

    frame_count = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret: break
            if frame is None: continue

            frame = cv2.LUT(frame, lut)
            n_pos, n_neg = noise_pool[frame_count % 30]
            cv2.add(frame, n_pos, frame)
            cv2.subtract(frame, n_neg, frame)

            proc.stdin.write(frame.tobytes())
            frame_count += 1
    except Exception as e:
        cap.release()
        proc.terminate()
        return False, str(e)

    cap.release()
    _stdout, stderr_bytes = proc.communicate()
    if proc.returncode != 0:
        return False, f"FFmpeg 오류: {stderr_bytes.decode(errors='ignore')}"
    return True, None

#### 3단계: 드라이브 연결 확인 및 단일 영상 변환 (`5e838cd7`, `4fb53fa7`)

*   **드라이브 연결 확인 (셀 5e838cd7):** 이 셀은 Google Drive가 올바르게 마운트되었는지 다시 한번 확인합니다. 이미 마운트되어 있으면 추가 마운트 시도를 피하고 메시지를 출력합니다.
*   **단일 영상 변환 및 저장 (셀 4fb53fa7):** 이 셀은 `apply_night_effect` 함수가 올바르게 작동하는지 테스트하기 위해 단일 동영상 파일에 야간 효과를 적용하고 저장하는 예시입니다.
    *   `input_video_path`: 원본 동영상 파일의 경로를 지정합니다.
    *   `output_directory`: 변환된 동영상이 저장될 폴더를 지정합니다.
    *   `BITRATE_FACTOR`: 비트레이트 조정 비율을 설정하여 파일 크기를 제어합니다 (예: 0.8은 20% 감소).
    *   `os.makedirs(output_directory, exist_ok=True)`: 출력 디렉토리가 없으면 생성합니다.
    *   `apply_night_effect` 함수를 호출하여 변환을 실행하고, 성공 또는 실패 메시지를 출력합니다.

In [ ]:
# 드라이브 연결을 다시 시도합니다.
# 이미 마운트되어 있어 오류가 발생할 경우 주석 처리하거나 무시하세요.
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive가 이미 마운트되어 있습니다. 연결을 새로 고치려면 런타임을 재시작한 후 이 셀을 실행하세요.")

# 마운트된 디렉토리 확인 (옵션)
# !ls -F /content/drive/MyDrive/


Google Drive가 이미 마운트되어 있습니다. 연결을 새로 고치려면 런타임을 재시작한 후 이 셀을 실행하세요.


### 단일 영상 변환 및 저장
사용자 요청에 따라 특정 영상을 변환하고 지정된 출력 경로에 저장합니다.

In [ ]:
import os

# Define input and output paths
input_video_path = "/content/drive/MyDrive/Dataset/Original/MP4/220510_LA_0004.mp4"
output_directory = "/content/drive/MyDrive/Dataset/Night"

# Adjust this value to control file size. 1.0 means no reduction, 0.8 means 20% reduction.
BITRATE_FACTOR = 0.8

# Ensure output directory exists
os.makedirs(output_directory, exist_ok=True)

# Construct output file path
output_video_filename = os.path.basename(input_video_path)
output_video_path = os.path.join(output_directory, output_video_filename)

print(f"원본 영상: {input_video_path}")
print(f"저장될 영상: {output_video_path}")
print(f"비트레이트 조정 비율: {BITRATE_FACTOR}")

# Apply night effect with bitrate factor
success, error_message = apply_night_effect(input_video_path, output_video_path, bitrate_factor=BITRATE_FACTOR)

if success:
    print(f"'{output_video_filename}' 영상 변환 성공 및 '{output_directory}'에 저장 완료.")
else:
    print(f"'{output_video_filename}' 영상 변환 실패: {error_message}")

드라이브가 성공적으로 다시 마운트되거나 확인되면, `a3f23d25` 셀을 다시 실행하여 비디오 처리 작업을 재시도해 보세요.

### 일괄 영상 변환 및 저장
여러 영상을 병렬로 변환하여 지정된 출력 폴더에 저장합니다.

In [ ]:
# --- Essential imports and environment setup for batch processing ---
import os
from tqdm.notebook import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Check if running in Google Colab to determine base directories
try:
    from google.colab import drive # Only import if in Colab environment
    # drive.mount('/content/drive') # Assuming drive is already mounted by a prior setup cell
    COLAB_ENV = True
except ImportError:
    COLAB_ENV = False

# --- End of essential setup ---

# NOTE: 이 스크립트는 'apply_night_effect' 함수가 정의되어 있어야 합니다.
# 'apply_night_effect' 함수가 포함된 셀 (예: Cell 2e1ed3ad)을 먼저 실행해 주세요.

def process_single_video(args):
    input_path, output_path, bitrate_factor = args
    filename = os.path.basename(input_path)
    try:
        success, error = apply_night_effect(input_path, output_path, bitrate_factor)
        if success:
            return filename, True, None
        else:
            if os.path.exists(output_path):
                try:
                    os.remove(output_path)
                except Exception as rm_e:
                    return filename, False, f"FFmpeg 오류: {error}, 출력 파일 제거 실패: {rm_e}"
            return filename, False, error
    except Exception as e:
        if os.path.exists(output_path):
            try:
                os.remove(output_path)
            except Exception as rm_e:
                return filename, False, f"예상치 못한 오류: {str(e)}, 출력 파일 제거 실패: {rm_e}"
        return filename, False, str(e)

def batch_convert_videos(bitrate_factor=0.8, input_base_dir=None, output_base_dir=None):
    if input_base_dir is None:
        input_base_dir = "/content/drive/MyDrive/Dataset/Original/MP4" if COLAB_ENV else "./Night/mp4"
    if output_base_dir is None:
        output_base_dir = "/content/drive/MyDrive/Dataset/Night" if COLAB_ENV else "./Night/output_mp4"

    if not os.path.exists(input_base_dir):
        print(f"입력 폴더를 찾을 수 없습니다: {input_base_dir}")
        return

    os.makedirs(output_base_dir, exist_ok=True)
    print(f"입력 폴더: {input_base_dir}")
    print(f"출력 폴더: {output_base_dir}")

    video_files = [f for f in os.listdir(input_base_dir) if f.endswith('.mp4')]
    if not video_files:
        print(f"{input_base_dir} 폴더에 처리할 MP4 영상 파일이 없습니다."); return

    tasks = []
    for f in video_files:
        input_path = os.path.join(input_base_dir, f)
        output_path = os.path.join(output_base_dir, f)
        tasks.append((input_path, output_path, bitrate_factor))

    # GPU 세션 제한을 방지하기 위해 max_workers를 4로 제한합니다.
    max_workers = 4
    print(f"GPU 세션 제한 방지를 위해 {max_workers}개의 프로세스를 사용하여 병렬 처리를 수행합니다.")

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_video, t): t for t in tasks}
        for f in tqdm(as_completed(futures), total=len(tasks), desc="영상 변환 작업 진행률"):
            res = f.result()
            if res[1]:
                print(f"\n[성공] {res[0]} 변환 완료.")
            else:
                print(f"\n[실패] {res[0]}: {res[2]}")

if __name__ == "__main__":
    batch_convert_videos(bitrate_factor=1.0)

입력 폴더: /content/drive/MyDrive/Dataset/Original/MP4
출력 폴더: /content/drive/MyDrive/Dataset/Night
GPU 세션 제한 방지를 위해 4개의 프로세스를 사용하여 병렬 처리를 수행합니다.


영상 변환 작업 진행률:   0%|          | 0/832 [00:00<?, ?it/s]


[성공] 220510_LA_0000.mp4 변환 완료.

[성공] 220510_LA_0007.mp4 변환 완료.

[성공] 220510_LA_0001.mp4 변환 완료.

[성공] 220510_LA_0002.mp4 변환 완료.

[성공] 220510_LA_0008.mp4 변환 완료.

[성공] 220510_LA_0003.mp4 변환 완료.

[성공] 220510_LA_0004.mp4 변환 완료.

[성공] 220510_LA_0005.mp4 변환 완료.

[성공] 220510_LA_0006.mp4 변환 완료.

[성공] 220510_LA_0011.mp4 변환 완료.

[성공] 220510_LA_0013.mp4 변환 완료.

[성공] 220510_LA_0010.mp4 변환 완료.

[성공] 220510_LA_0014.mp4 변환 완료.

[성공] 220510_LA_0009.mp4 변환 완료.

[성공] 220510_LA_0015.mp4 변환 완료.

[성공] 220510_LA_0012.mp4 변환 완료.

[성공] 220510_LA_0020.mp4 변환 완료.

[성공] 220510_LA_0017.mp4 변환 완료.

[성공] 220510_LA_0019.mp4 변환 완료.

[성공] 220510_LA_0016.mp4 변환 완료.

[성공] 220510_LA_0018.mp4 변환 완료.

[성공] 220510_LA_0023.mp4 변환 완료.

[성공] 220510_LA_0024.mp4 변환 완료.

[성공] 220510_LA_0022.mp4 변환 완료.

[성공] 220510_LA_0021.mp4 변환 완료.

[성공] 220510_LA_0025.mp4 변환 완료.

[성공] 220510_LA_0029.mp4 변환 완료.

[성공] 220510_LA_0028.mp4 변환 완료.

[성공] 220510_LA_0026.mp4 변환 완료.

[성공] 220510_LA_0027.mp4 변환 완료.

[성공] 220510_LA_0032.mp4 변환 완료.

[성공] 22

### 전체 작업 흐름 요약

1.  **환경 준비**: Google Drive를 마운트하여 데이터 접근 경로를 설정합니다.
2.  **핵심 기능 정의**: `apply_night_effect` 함수를 통해 단일 동영상에 야간 효과(감마 보정 + 노이즈)를 적용하는 로직을 정의합니다. 이때, `ffprobe`를 사용하여 원본 비트레이트를 분석하고 출력 비트레이트를 조절하여 파일 크기를 최적화합니다.
3.  **단일 테스트**: 정의된 `apply_night_effect` 함수를 사용하여 하나의 동영상에 효과를 적용해보고 정상 작동하는지 확인합니다.
4.  **병렬 처리**: `batch_convert_videos` 함수는 `apply_night_effect` 함수를 여러 동영상에 대해 `ProcessPoolExecutor`를 활용하여 병렬로 실행합니다. 이는 다수의 동영상 파일을 효율적으로 처리하는 데 중요합니다.